In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="PR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="PR")


all_loss = {}
for epoch in range(1000):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().numpy().item()))

epoch:  0, loss: 0.1343381255865097
epoch:  1, loss: 0.09183039516210556
epoch:  2, loss: 0.06832990050315857
epoch:  3, loss: 0.055096760392189026
epoch:  4, loss: 0.04756023734807968
epoch:  5, loss: 0.04323667660355568
epoch:  6, loss: 0.040743838995695114
epoch:  7, loss: 0.03930095210671425
epoch:  8, loss: 0.03846301510930061
epoch:  9, loss: 0.03797483816742897
epoch:  10, loss: 0.03768919035792351
epoch:  11, loss: 0.03752104938030243
epoch:  12, loss: 0.03742131218314171
epoch:  13, loss: 0.03736145421862602
epoch:  14, loss: 0.037324994802474976
epoch:  15, loss: 0.037302203476428986
epoch:  16, loss: 0.03729179874062538
epoch:  17, loss: 0.037269044667482376
epoch:  18, loss: 0.037254832684993744
epoch:  19, loss: 0.03724868595600128
epoch:  20, loss: 0.037234336137771606
epoch:  21, loss: 0.03722545504570007
epoch:  22, loss: 0.037222281098365784
epoch:  23, loss: 0.037212878465652466
epoch:  24, loss: 0.03721189871430397
epoch:  25, loss: 0.037201814353466034
epoch:  26, l

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8365080506538165
Test metrics:  R2 = 0.8143598302845478


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="FR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="FR")

all_loss = {}
for epoch in range(1000):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().numpy().item()))

epoch:  0, loss: 0.24003268778324127
epoch:  1, loss: 0.1479485183954239
epoch:  2, loss: 0.09827730804681778
epoch:  3, loss: 0.0709688812494278
epoch:  4, loss: 0.055857133120298386
epoch:  5, loss: 0.04748512804508209
epoch:  6, loss: 0.042851317673921585
epoch:  7, loss: 0.04028989002108574
epoch:  8, loss: 0.03887426480650902
epoch:  9, loss: 0.03809042274951935
epoch:  10, loss: 0.03765415772795677
epoch:  11, loss: 0.037408486008644104
epoch:  12, loss: 0.037266794592142105
epoch:  13, loss: 0.03718218952417374
epoch:  14, loss: 0.03715553879737854
epoch:  15, loss: 0.03704879432916641
epoch:  16, loss: 0.03698733448982239
epoch:  17, loss: 0.036950770765542984
epoch:  18, loss: 0.03693532943725586
epoch:  19, loss: 0.0368875153362751
epoch:  20, loss: 0.0368589423596859
epoch:  21, loss: 0.0368470773100853
epoch:  22, loss: 0.036808330565690994
epoch:  23, loss: 0.036784958094358444
epoch:  24, loss: 0.03676830232143402
epoch:  25, loss: 0.036736492067575455
epoch:  26, loss: 0

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7747147296759703
Test metrics:  R2 = 0.7287945846204356
